In [1]:
!pip install bitsandbytes

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


# MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
# 4 bit 
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,              # Enable 4-bit loading
#     bnb_4bit_use_double_quant=True, # Use nested quantization to save more memory
#     bnb_4bit_quant_type="nf4",      # Normal Float 4 (optimal for LLMs)
#     bnb_4bit_compute_dtype=torch.float16 # Compute in float16 for speed
# )

def load_model():
    print(f"Loading {MODEL_ID}...")
    
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    
    # 1. Ensure EOS token is valid
    if tokenizer.eos_token_id is None:
        if hasattr(tokenizer, "eod_id"):
             tokenizer.eos_token = tokenizer.decode(tokenizer.eod_id)
        else:
             tokenizer.add_special_tokens({'eos_token': '<|endoftext|>'})

    # 2. Ensure PAD token is valid
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        
    print(f"Tokenizer Pad ID: {tokenizer.pad_token_id}")
    print(f"Tokenizer EOS ID: {tokenizer.eos_token_id}")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        # quantization_config=bnb_config,
        device_map="auto", 
        trust_remote_code=True
    )
    
    # 3. Synchronize Model Configuration
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id
    
    return model, tokenizer

def generate_response(prompt, model, tokenizer):
    system_prompt = """You are an Aspect-Based Sentiment Analysis assistant. 
Given a Text and an Aspect term, determine the Sentiment (Positive, Negative, or Neutral) and extract the specific Opinion word or phrase describing that aspect.

Format your response exactly as:
Sentiment: <Positive/Negative/Neutral>
Opinion: <Opinion Phrase>

Examples:
Text: The food was great but the service was terrible.
Aspect term: food
Sentiment: Positive
Opinion: great

Text: The food was great but the service was terrible.
Aspect term: service
Sentiment: Negative
Opinion: terrible

Text: Овощной салат также пришёлся по вкусу.
Aspect term: Овощной салат
Sentiment: Positive
Opinion: пришёлся по вкусу"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt}
    ]
    
    input_ids = tokenizer.apply_chat_template(
        messages, 
        add_generation_prompt=True, 
        return_tensors="pt"
    ).to(model.device)


    terminators = [tokenizer.eos_token_id]
    try:
        eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
        if isinstance(eot_id, int):
            terminators.append(eot_id)
    except:
        pass

    outputs = model.generate(
        input_ids,
        max_new_tokens=64,   # Increased to accommodate Sentiment + Opinion
        eos_token_id=terminators,
        pad_token_id=tokenizer.pad_token_id, 
        do_sample=True,      
        temperature=0.3,     # Lower temperature for more consistent formatting
        top_p=0.9,
    )

    
    response = outputs[0][input_ids.shape[-1]:]
    return tokenizer.decode(response, skip_special_tokens=True)

In [3]:
model, tokenizer = load_model()

Loading Qwen/Qwen3-4B-Instruct-2507...


Tokenizer Pad ID: 151643
Tokenizer EOS ID: 151645


2026-02-13 16:22:40.807464: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770999760.831449   88945 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770999760.836540   88945 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770999760.850427   88945 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770999760.850443   88945 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770999760.850446   88945 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
import json
import copy
import os
from tqdm import tqdm

def parse_raw_data(raw_data_input):
    """
    Parses input which can be:
    1. A raw string containing line-separated JSON objects.
    2. A file path (str) ending in .jsonl or .json.
    """
    data = []
    
    
    if isinstance(raw_data_input, str) and os.path.exists(raw_data_input):
        print(f"Reading from file: {raw_data_input}")
        with open(raw_data_input, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        data.append(json.loads(line))
                    except json.JSONDecodeError:
                        print(f"Skipping invalid JSON line in file.")
        return data

    
    lines = raw_data_input.strip().split('\n')
    for line in lines:
        if line.strip():
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError:
    
                print(f"Skipping invalid JSON line: {line[:50]}...")
    return data

def batch_inference(model, tokenizer, data, batch_size=4):
    """
    Runs inference in batches and updates a copy of the dataset with the results.
    Returns: A new list of data entries with Gen_Sentiment and Gen_Opinion fields added.
    """
    
    processed_data = copy.deepcopy(data)
    
    
    work_items = []
    
    for entry in processed_data:
        text = entry.get('Text', '')
        
        
        if 'Aspect_VA' in entry:
            for item in entry['Aspect_VA']:
                work_items.append({
                    "text": text,
                    "aspect": item['Aspect'],
                    "container": item # Reference to dict inside processed_data
                })
        

        elif 'Quadruplet' in entry:
            for item in entry['Quadruplet']:
                aspect = item.get('Aspect', 'NULL')
                if aspect == "NULL": 
                    aspect = item.get('Category', 'general').split('#')[0]
                
                work_items.append({
                    "text": text,
                    "aspect": aspect,
                    "container": item
                })
                
    print(f"Processing {len(work_items)} aspect items in batches of {batch_size}...")
    

    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
         tokenizer.pad_token = tokenizer.eos_token

    system_prompt = """You are an Aspect-Based Sentiment Analysis assistant. 
Given a Text and an Aspect term, determine the Sentiment (Positive, Negative, or Neutral) and extract the specific Opinion word or phrase describing that aspect.

Format your response exactly as:
Sentiment: <Positive/Negative/Neutral>
Opinion: <Opinion Phrase>

Examples:
Text: The food was great but the service was terrible.
Aspect term: food
Sentiment: Positive
Opinion: great

Text: Овощной салат также пришёлся по вкусу.
Aspect term: Овощной салат
Sentiment: Positive
Opinion: пришёлся по вкусу"""


    for i in tqdm(range(0, len(work_items), batch_size), desc="Inferencing"):
        batch = work_items[i:i + batch_size]
        batch_prompts = []
        
        for item in batch:
            user_msg = f"Text: {item['text']}\nAspect term: {item['aspect']}"
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_msg}
            ]
            prompt_text = tokenizer.apply_chat_template(
                messages, 
                tokenize=False, 
                add_generation_prompt=True
            )
            batch_prompts.append(prompt_text)
            
        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        ).to(model.device)
        
        terminators = [tokenizer.eos_token_id]
        if hasattr(tokenizer, "convert_tokens_to_ids"):
             try:
                eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
                if isinstance(eot, int): terminators.append(eot)
             except: pass

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=64,
                eos_token_id=terminators,
                pad_token_id=tokenizer.pad_token_id,
                do_sample=True,
                temperature=0.3,
                top_p=0.9
            )
            
        generated_ids = outputs[:, inputs.input_ids.shape[1]:]
        decoded_texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        

        for work_item, response in zip(batch, decoded_texts):
            response = response.strip()
            

            sentiment = "Unknown"
            opinion = "Unknown"
            
            lines = response.split('\n')
            for line in lines:
                if line.startswith("Sentiment:"):
                    sentiment = line.split("Sentiment:", 1)[1].strip()
                elif line.startswith("Opinion:"):
                    opinion = line.split("Opinion:", 1)[1].strip()
            

            work_item['container']['Gen_Sentiment'] = sentiment
            work_item['container']['Gen_Opinion'] = opinion
            
    return processed_data

In [5]:
user_prompt = "Text:Кухня в ЗимаЛето очень неплохая: хорошее и вкусное гриль-меню (особенно рыба), стандартный выбор салатиков, большой выбор суши и роллов (самые вкусные Гейша и запеченные Киото), достойная коктельная карта, только вот лонг, оставляет желать лучшего, а клубничную маргариту не советую никому, отвратная! Aspect:клубничную маргариту"
result = generate_response(user_prompt, model, tokenizer)
print(result)   

Sentiment: Negative  
Opinion: отвратная!


In [ ]:
raw_data_str = """
{"ID":"36:2_0","Text":"Что могу сказать, посидели чисто по русски в итальянском ресторане :)","Aspect_VA":[{"Aspect":"ресторане","VA":"7.17#6.33"}]}
{"ID":"36:3_1","Text":"Место достаточно комфортное, есть диванчики.","Aspect_VA":[{"Aspect":"Место","VA":"6.5#5.25"},{"Aspect":"диванчики","VA":"6.5#5.5"}]}
{"ID":"36:4_2","Text":"Играла приятная музыка, что также порадовало.","Aspect_VA":[{"Aspect":"музыка","VA":"6.67#5.67"}]}
{"ID":"36:5_3","Text":"Делали много заказов, мой выбор пал на пасту с копчёной грудкой.","Aspect_VA":[{"Aspect":"пасту с копчёной грудкой","VA":"6.0#4.5"}]}
{"ID":"36:7_4","Text":"Овощной салат также пришёлся по вкусу.","Aspect_VA":[{"Aspect":"Овощной салат","VA":"6.83#5.5"}]}
{"ID":"36:8_5","Text":"Мясная и сырная тарелки вполне себе обычные, но что-то негативное сказать про них нельзя.","Aspect_VA":[{"Aspect":"Мясная и сырная тарелки","VA":"5.0#2.33"}]}
{"ID":"36:9_6","Text":"Ребята заказывали в основном стейки из мяса и лосося, и по их довольным лицам можно было понять, что еда определённо вкусная и доставляет удовольствие.","Aspect_VA":[{"Aspect":"стейки из мяса и лосося","VA":"7.67#6.92"}]}
{"ID":"36:10_7","Text":"Не хватало лишь комплимента от шеф-повара, но это так, придирки.","Aspect_VA":[{"Aspect":"комплимента от шеф-повара","VA":"3.0#6.75"}]}
{"ID":"36:12_8","Text":"Тут никаких сюрпризов, разве что ассортимент пива можно сделать и больше, не как в пабах, но 3-4 варианта самое то.","Aspect_VA":[{"Aspect":"пива","VA":"3.75#5.25"}]}
{"ID":"36:14_9","Text":"Атмосфера в ресторане приятная, располагает к отдыху.","Aspect_VA":[{"Aspect":"Атмосфера","VA":"7.29#6.54"}]}
{"ID":"36:17_10","Text":"Обслуживание оказалось на высоте, вполне возможно, что из-за обозначенных нами амбиций на объём заказа и как следствие будущий чай. не знаю.","Aspect_VA":[{"Aspect":"Обслуживание","VA":"7.67#6.5"}]}
{"ID":"36:18_11","Text":"Но девушка оказалась очень доброй, внимательной и просто красивой :)","Aspect_VA":[{"Aspect":"девушка","VA":"7.11#6.06"}]}
{"ID":"151:3_12","Text":"Еда была очень вкусная и разнообразная, сервис на высшем уровне!!","Aspect_VA":[{"Aspect":"Еда","VA":"7.25#6.25"},{"Aspect":"сервис","VA":"8.17#8.0"}]}
{"ID":"151:4_13","Text":"Еды кстати было очень много, что в конце вечера оставшееся нам завернули с собой.","Aspect_VA":[{"Aspect":"Еды","VA":"7.33#6.33"}]}
"""

parsed_data = parse_raw_data("/kaggle/working/dimabsa/initial_data_train/eng_laptop_train_alltasks.jsonl")
print(f"Parsed {len(parsed_data)} entries.")

updated_dataset = batch_inference(model, tokenizer, parsed_data, batch_size=4)

print("\n--- Processed Dataset Results ---")
for entry in updated_dataset[:5]:
    print(entry)

print("\n--- Clean View ---")
for entry in updated_dataset[:5]:
    if 'Aspect_VA' in entry:
        for item in entry['Aspect_VA']:
            print(f"Aspect: {item['Aspect']}")
            print(f"  -> Gen Sentiment: {item.get('Gen_Sentiment')}")
            print(f"  -> Gen Opinion:   {item.get('Gen_Opinion')}")

Reading from file: /kaggle/working/dimabsa/initial_data_train/eng_laptop_train_alltasks.jsonl
Parsed 4076 entries.
Processing 5773 aspect items in batches of 4...


Inferencing:   0%|          | 0/1444 [00:00<?, ?it/s]

Inferencing:  16%|█▋        | 236/1444 [15:48<1:20:53,  4.02s/it]


KeyboardInterrupt: 

In [ ]:

dataset = parse_raw_data(updated_dataset)
print(f"Loaded {len(dataset)} entries for inference.")


final_results = batch_inference(model, tokenizer, dataset, batch_size=4)


print("\n--- Final Output (JSON) ---")
print(json.dumps(final_results, indent=2, ensure_ascii=False))


AttributeError: 'list' object has no attribute 'strip'